## LLama3.2 Fine-tuning
In this notebook, we will fine-tune Llama 3.2 on the ROCOv2 dataset. Let's get started by installing necessary libraries.

In [ ]:
%pip install -U tensorboard-plugin-profile

In [ ]:
%pip install -U transformers datasets unsloth

## Configuration and imports

In [ ]:
#HF_HOME="/kaggle/working/huggingface" #@param {type:"string"}
HF_TOKEN="<insert your HF token here>" #@param {type:"string"}
#WB_TOKEN="<insert your wandb token here>" #@param {type:"string"}

In [ ]:
import os
#os.environ["HF_HOME"] = HF_HOME
os.environ["HF_TOKEN"] = HF_TOKEN
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from datetime import datetime
#import wandb

In [ ]:
import multiprocessing

In [ ]:
import os
from unsloth import FastVisionModel
import torch
from datasets import load_dataset
from transformers import TextStreamer
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [ ]:
#wandb.login(key=WB_TOKEN)
#run = wandb.init(project='Fine-tune Llama3.2-11B on RocoV2', job_type="training", anonymous="allow")

## Load the model
Note: we are using 4-bit quantization

In [ ]:
# 1. Load the model

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    #device_map = "auto",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules      = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## Load the dataset

In [ ]:
from datasets import load_dataset
train_ds = load_dataset('eltorio/ROCOv2-radiology', trust_remote_code=True, split="train").remove_columns(['image_id', 'cui'])
test_ds = load_dataset('eltorio/ROCOv2-radiology', trust_remote_code=True, split="test").remove_columns(['image_id', 'cui'])
eval_ds = load_dataset('eltorio/ROCOv2-radiology', trust_remote_code=True, split="validation").remove_columns(['image_id', 'cui'])


In [ ]:
train_ds[5]

In [ ]:
train_ds

## Preprocess the dataset

**Note**: Using the 'map' function for HF datasets for whatever reason reads the whole base-64-encoded image into each input element and then returns messages in an incorrect format. We need to write a special data collator.

In [ ]:
prompt_system = "You are an expert radiographer certified with over 15 years of experience in diagnostic imaging. Respond in a formal and objective tone, using technical radiography terminology. The images may include X-rays, CT scans, MRI scans, ultrasounds, or other diagnostic imaging modalities. Your mission is to provide a thorough and accurate description of the image, including any notable features, abnormalities, or diagnostic findings."
prompt_user = "Please describe accurately this image."
instruction = prompt_system+"\n"+prompt_user

**Note:** Alas LLama 3.2 Multimodal does not accept system prompts, so we have to provide all instructions in the user prompt.

In [ ]:
def convert_to_conversation(message):
    conversation = [
        #{ "role": "system",
        #  "content" : [
        #    {"type" : "text",  "text"  : prompt_system},
        #    {"type" : "image", "image" : message["image"]} ]
        #},
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : message["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : message["caption"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator

class MyUnslothVisionDataCollator(UnslothVisionDataCollator):
    def __call__(self, messages):
        # messages is a list of Dataset elements
        preprocessed_examples = self.preprocess_examples(messages)
        return super().__call__(preprocessed_examples)
    
    def preprocess_examples(self, messages):
        return [self.convert_to_conversation(message) for message in messages]
    
    def convert_to_conversation(self, message):
        conversation = [
            #{ "role": "system",
            #"content" : [
            #    {"type" : "text",  "text"  : prompt_system},
            #    {"type" : "image", "image" : message["image"]} ]
            #},
            { "role": "user",
            "content" : [
                {"type" : "text",  "text"  : instruction},
                {"type" : "image", "image" : message["image"]} ]
            },
            { "role" : "assistant",
            "content" : [
                {"type" : "text",  "text"  : message["caption"]} ]
            },
        ]
        return { "messages" : conversation }
pass

In [ ]:
## Test Collator ##
collator = MyUnslothVisionDataCollator(model, tokenizer)
# Create a list of 10 dataset elements
minilist = []
for i in range(10):
    minilist.append(train_ds[i])

collator(minilist)

## Training
### Before

In [ ]:
# for speed and memory limitations, we have to use a small sample. the model won't be very good.
SAMPLE_SIZE = 1000 #@param {"type": "integer"} 

In [ ]:
test_ds[456]["image"]

In [ ]:
test_ds[456]["caption"]

In [ ]:
# 3. Before training

FastVisionModel.for_inference(model)
image = test_ds[456]["image"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]



In [ ]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")



In [ ]:
print("\nBefore training:\n")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

### Configure

In [ ]:
# 4. Training

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = MyUnslothVisionDataCollator(model, tokenizer), # Must use!
    #data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_ds.take(SAMPLE_SIZE),
    eval_dataset = eval_ds.take(SAMPLE_SIZE//10),
    args = SFTConfig(
        num_train_epochs=1,
        auto_find_batch_size = True,
        #per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        #max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 10,
        save_steps = 100,
        eval_strategy="steps",
        eval_steps = 100,
        save_total_limit = 3,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/llama32-rocov2",
        report_to = "tensorboard",
        run_name="llama32-rocov2"+str(datetime.now()),
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = multiprocessing.cpu_count(),
        max_seq_length = 2048,
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties()
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

### Train

In [ ]:
trainer_stats = trainer.train()

In [ ]:
#wandb.finish()

In [ ]:
# Load the TensorBoard notebook extension if available
%load_ext tensorboard

In [ ]:
#wandb.finish()
%tensorboard --logdir /content/llama32-rocov2

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### After

In [ ]:
# 5. After training

print("\nAfter training:\n")
FastVisionModel.for_inference(model)
image = test_ds[456]["image"]
instruction = "You are an expert radiologist. Analyze and diagnose what you see in this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
test_ds[456]

In [ ]:
test_ds[456]["image"]

## Save the model

In [ ]:
# 6. Save the model

model.save_pretrained("llama32-rocov2-lora")
tokenizer.save_pretrained("llama32-rocov2-lora")

## OPTIONAL: Push to HF hub
#model.save_pretrained_merged("gimarchetti/Llama-3.2-11B-Vision-RocoV2-4bit", tokenizer,)
#model.push_to_hub_merged("gimarchetti/Llama-3.2-11B-Vision-RocoV2-4bit", tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))

## Evaluation

In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor, LlavaForConditionalGeneration

In [ ]:
model_id = "<insert your HF model id here>" # or use the pretrained "gimarchetti/Llama-3.2-11B-Vision-RocoV2-4bit"
processor_id="meta-llama/Llama-3.2-11B-Vision-Instruct"
processor = AutoProcessor.from_pretrained(processor_id)
model = AutoModelForVision2Seq.from_pretrained(model_id,
                                               torch_dtype=torch.bfloat16,
                                               #_attn_implementation="flash_attention_2",
                                               device_map="auto")

In [ ]:
model.eval()

In [ ]:
example = test_ds[752]
example

In [ ]:
example["image"]

In [ ]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
image=example["image"].convert("RGB")#.resize((256, 256))
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to(model.device)

output = model.generate(**inputs, max_new_tokens=128)
print(processor.decode(output[0]))

## Evaluate on subset

In [ ]:
example

In [ ]:
def generate_caption(example, max_new_tokens=32, temperature=0.5, top_p=0.9, top_k=40):
  image = example["image"].convert("RGB")#.resize((256, 256))
  messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
  ]
  input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
  inputs=processor(
      image,
      input_text,
      add_special_tokens=False,
      return_tensors="pt",
      #padding="longest",
    ).to("cuda")
  generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p, top_k=top_k, do_sample=True)
  #print(generated_ids
  generated_texts= processor.decode(generated_ids[0], skip_special_tokens=True).split("\n")[-1]
  return generated_texts


In [ ]:
generate_caption(example, temperature=0.4, top_p=0.5, top_k=10)

In [ ]:
SAMPLE_SIZE = 5 #@param

In [ ]:
from matplotlib import pyplot as plt

fig = plt.figure(figsize=(20, 20 ))# 5*SAMPLE_SIZE))

# prepare image for the model
for i, example in enumerate(test_ds.shuffle().take(SAMPLE_SIZE)):
  generated_caption = generate_caption(example, max_new_tokens=32, temperature=0.4, top_p=0.5, top_k=10 )
  fig.add_subplot(SAMPLE_SIZE, 1, i+1)
  plt.imshow(example["image"])
  plt.subplots_adjust(hspace=0.1*SAMPLE_SIZE)  # Adjust vertical spacing
  #plt.axis("off")
  plt.title(f"Generated caption: {generated_caption}")
  plt.xlabel(f"Original caption: {example['text']}")

In [ ]:
model.save_pretrained("/kaggle/working/llama32-rocov2")